# ATSC-411 - SYNOPTIC METEOROLOGY | PV LAB


**************

## IMPORT LIBRARIES


In [ ]:
# import packages
import numpy as np                               # numpy for math & number operations
import cartopy.crs as ccrs                       # cartopy coordinate refrence system for mapping -- to turn a matplotlib axis into a geoaxis (map)
import cartopy.feature as cfeature               # cartopy features (borders, states, oceans, rivers, etc) for mapping -- add map components to a map
import pyproj
import matplotlib.pyplot as plt                  # matplotlib for plotting figures, axes, graphs, etc
from matplotlib.colors import LinearSegmentedColormap

import helper_fcns as helper
import metpy.calc as mpcalc
from metpy.units import units
from metpy.interpolate import cross_section
from metpy.interpolate import interpolate_to_isosurface

from herbie import Herbie

print("[+] packages imported...")

**************

## LOAD MODEL DATA FOR A GIVEN DATE

In [ ]:
# date/time components
year  = "2022"
month = "04"
day   = "12"
hour  = "12"
fxx = 0
#options are GFS and NAM
model = 'gfs'

date2grab = f"{year}-{month}-{day} {hour}:00"

if model=="nam":
    H = Herbie(date2grab,
               model = model,
               product='awphys',
               fxx = fxx,
              )
else:
    H = Herbie(date2grab,
               model = model,
               fxx = fxx,
              )

ds = helper.get_xarray(H,model)

**************

## Compute plan-view map variables
#### Compute potential temperature, lat/lon grid deltas, potential vorticity, theta and wind on a 2PVU isosurface and geopotential hgt slices

In [ ]:
#################################
# COMPUTE THETA AND PV
################################# 

ds['wind_speed'] = mpcalc.wind_speed(ds.u, ds.v)

# use metpy to compute theta & add it into the `rap-data` DataSet
ds['theta'] = mpcalc.potential_temperature(ds['isobaricInhPa'],ds['t'])

# use metpy to compute latitude / longitude grid deltas (dx, dy) for PV calculation
dx, dy = mpcalc.lat_lon_grid_deltas(ds['longitude'].values, ds['latitude'].values)
dx = dx[None, :, :]
dy = dy[None, :, :]

ds['pv'] = mpcalc.potential_vorticity_baroclinic(potential_temperature=ds['theta'],
                                                 pressure=ds['isobaricInhPa'],
                                                 u=ds['u'],
                                                 v=ds['v'],
                                                 dx=dx, dy=dy,
                                                 latitude=ds['latitude'],)

In [ ]:
#################################
# Get Theta, Wind, and Pressure at 2 PVU Surface
################################# 

#need to make pressure 3-D
p_3d = np.zeros(ds['pv'].shape)
for i in range(len(p_3d[0])):
    for j in range(len(p_3d[0,i])):
        p_3d[:,i,j] = ds['isobaricInhPa'].values

#ignore things at 800mb or closer to the surface as noise in the midlats
idx = np.where(ds['isobaricInhPa'].values<800)[0][0]

thta_2pvu = interpolate_to_isosurface(ds['pv'][idx:,:,:].values, ds['theta'][idx:,:,:].values,  2*1e-6, bottom_up_search=True)
u_2pvu    = interpolate_to_isosurface(ds['pv'][idx:,:,:].values, ds['u'][idx:,:,:].values, 2*1e-6, bottom_up_search=True)
v_2pvu    = interpolate_to_isosurface(ds['pv'][idx:,:,:].values, ds['v'][idx:,:,:].values, 2*1e-6, bottom_up_search=True)
p_2pvu    = interpolate_to_isosurface(ds['pv'][idx:,:,:].values, p_3d[idx:,:,:], 2*1e-6, bottom_up_search=True)


In [ ]:
#################################
# DEFINE USEFUL PRESSURE LEVEL SLICES
################################# 
# create variables that hold the "index" (location) of a given height in the isobaric dimension
# i.e., 300 hPa is the 8th value in the isobaric dimension, use it to slice other variables
plev250 = list(ds['isobaricInhPa']).index(((250 * units('hPa')).to(ds['isobaricInhPa'].units)).m)
plev300 = list(ds['isobaricInhPa']).index(((300 * units('hPa')).to(ds['isobaricInhPa'].units)).m)
plev500 = list(ds['isobaricInhPa']).index(((500 * units('hPa')).to(ds['isobaricInhPa'].units)).m)



In [ ]:
#################################
# CREATE A TIDY DATE STRING
################################# 
try:
    data_date = ds['time'].values
except:
    data_date = ds['time1'].values
    pass
valid_date = f'{data_date}'

**************

## Define the `build_map()`, & `build_inset_map()` functions & create custom colormaps

In [ ]:
#--------------------------------------------------------------------------------------
# BUILD MAPS FUNCTION -----------------------------------------------------------------
#--------------------------------------------------------------------------------------
# define build maps func
def build_map(extent=[-121, -73, 21, 56]):
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=ccrs.LambertConformal())

    # apply the map extent (lat/lon bounding box)
    ax.set_extent(extent)
    # axis aspect ratio
    ax.set_box_aspect(0.7)
    # add map features
    ax.add_feature(cfeature.STATES, edgecolor='navy', alpha=0.5, linestyle='-', linewidth=1, zorder=10)
    ax.add_feature(cfeature.BORDERS, color='navy', alpha=1, linestyle='-', linewidth=1, zorder=11)
    ax.add_feature(cfeature.COASTLINE, color='navy', alpha=0.5, linestyle='-', linewidth=1, zorder=11)

    # apply tight layout to the figure (keeps things tiddy)
    plt.tight_layout()

    # return the figure axis
    return fig, ax


#--------------------------------------------------------------------------------------
# BUILD INSET MAP FUNCTION -----------------------------------------------------------------
#--------------------------------------------------------------------------------------
def build_inset_map(extent=[-121, -73, 21, 56]):

    # SET UP MAP INSET AXIS
    proj = ccrs.LambertConformal()
    ax = fig.add_axes([0.73, 0.712, 0.17, 0.17], projection=proj)
    ax.set_box_aspect(0.7)
    ax.set_extent(extent, ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE, edgecolor='navy', linewidth=0.5)
    ax.add_feature(cfeature.STATES, edgecolor='navy', linewidth=0.5)

    return ax

#--------------------------------------------------------------------------------------
# DEFINE COLORMAPS    -----------------------------------------------------------------
#--------------------------------------------------------------------------------------

# wind speed colormap
wdsp_colors = [(0.000, '#FFFFFF'),(0.004, '#87CEFA'),(0.166, '#6A5ACD'),(0.250, '#E696DC'),(0.333, '#C85ABE'),
          (0.416, '#A01496'), (0.500, '#C80028'), (0.583, '#DC283C'), (0.666, '#F05050'),(0.750, '#FAF064'),
          (0.833, '#DCBE46'), (0.916, '#BE8C28'),(1.000, '#A05A0A')]
wdsp_cmap = LinearSegmentedColormap.from_list("custom_cmap",wdsp_colors,N=256)

# potential vorticity colormap
pv_clevs  = np.append(np.arange(-1.4, 2, 0.2),np.arange(2,10.2,0.4))
pv_colors = ["blue","lightblue","lightblue","yellow","orange","red","darkred"]
pv_cmap   = LinearSegmentedColormap.from_list("custom_cmap",pv_colors)

# theta on the 2PVU surface colormap
theta_2pvu_colors = ["purple","lightblue","blue","green","yellow","red","darkred"]
theta_2pvu_cmap = LinearSegmentedColormap.from_list("custom_cmap", theta_2pvu_colors)

print("\n[+] functions and colormaps created...")


**************

## 300hPa plan view map of geopotential height and wind

In [ ]:
#################################
# BUILD AXIS AND FIGURE
#################################
fig, ax = build_map()

# define the level to slice
slice = plev300

if model=='gfs':
    every = 1
    lat=ds['latitude'].values[::every]
    lon=ds['longitude'].values[::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every]
    wind_lon=ds['longitude'].values[::wind_every]
    
elif model=='nam':
    every = 1
    lat=ds['latitude'].values[::every, ::every]
    lon=ds['longitude'].values[::every, ::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every, ::wind_every]
    wind_lon=ds['longitude'].values[::wind_every, ::wind_every]
    
#################################
# PLOT DATA ON THE MAP
#################################
contourf = ax.contourf(lon, lat, ds['wind_speed'][slice,::every,::every] * 1.94384, np.arange(40,160,5), cmap=wdsp_cmap,
                 zoder=4,transform=ccrs.PlateCarree(),extend='both')

# plot geopotential height contours
contour = ax.contour(ds['longitude'], ds['latitude'], ds['gh'][slice,:,:], np.arange(0, 12000, 60),
                colors='black', linewidths=3.0, linestyles='-',
                transform=ccrs.PlateCarree(), zorder=11)
plt.clabel(contour, fontsize=8, inline=1, inline_spacing=10, fmt='%i',
           rightside_up=True, use_clabeltext=True)

barbs = ax.barbs(wind_lon, wind_lat,
                 ds['u'][slice][::wind_every,::wind_every].values * 1.94384, 
                 ds['v'][slice][::wind_every,::wind_every].values * 1.94384,
                 length=8, alpha=1.0, transform=ccrs.PlateCarree(), zorder=12)


# add some plot titles
plt.title('  300-hPa GFS 300-hPa Geopotential Height (dam),\n'
          '  and Wind Barbs (kt)', loc='left', fontsize=20)
plt.title(f'{valid_date[0:10]} {valid_date[11:-13]}UTC  ', loc='right', fontsize=20)


# colorbar for filled contour
cbar = plt.colorbar(contourf, aspect=70, fraction=0.02, ax=ax, orientation='horizontal', pad=-0.01, extendrect=True)
cbar.set_label(r'Wind Speed (kts)', fontsize=15)

print("\n[+] plotting...")

# Plot a simple 300hPa PV map
### Geopotential Height, Wind, Potential Vorticity

In [ ]:
#################################
# BUILD AXIS AND FIGURE
#################################
fig, ax = build_map()

# define the level to slice
slice = plev300
#
if model=='gfs':
    every = 1
    lat=ds['latitude'].values[::every]
    lon=ds['longitude'].values[::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every]
    wind_lon=ds['longitude'].values[::wind_every]
    
elif model=='nam':
    every = 1
    lat=ds['latitude'].values[::every, ::every]
    lon=ds['longitude'].values[::every, ::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every, ::wind_every]
    wind_lon=ds['longitude'].values[::wind_every, ::wind_every]
    
#################################
# PLOT DATA ON THE MAP
#################################
contourf = ax.contourf(lon, lat, ds['pv'][slice,::every,::every]*1e6, pv_clevs, cmap=pv_cmap,
                 transform=ccrs.PlateCarree(),extend='both')

# plot a single dashed contour @ 2PVU
pv_contour = ax.contour(lon, lat, ds['pv'][slice][::every,::every]*1e6, [2], colors='navy',linestyles='dashed',linewidths=2,
                 transform=ccrs.PlateCarree())


# plot geopotential height contours
contour = ax.contour(ds['longitude'], ds['latitude'], ds['gh'][slice,:,:], np.arange(0, 12000, 60),
                colors='black', linewidths=3.0, linestyles='-',
                transform=ccrs.PlateCarree(), zorder=11)
plt.clabel(contour, fontsize=8, inline=1, inline_spacing=10, fmt='%i',
           rightside_up=True, use_clabeltext=True)

barbs = ax.barbs(wind_lon, wind_lat,
                 ds['u'][slice][::wind_every,::wind_every].values * 1.94384, 
                 ds['v'][slice][::wind_every,::wind_every].values * 1.94384,
                 length=8, alpha=1.0, transform=ccrs.PlateCarree(), zorder=12)

# add some plot titles
plt.title('  300-hPa GFS PV (PVU), 300-hPa Geopotential Height (dam),\n'
          '  and Wind Barbs (kt)', loc='left', fontsize=20)
plt.title(f'{valid_date[0:10]} {valid_date[11:-13]}UTC  ', loc='right', fontsize=20)

# colorbar for filled contour
cbar = plt.colorbar(contourf, aspect=70, fraction=0.02, ax=ax, orientation='horizontal', pad=-0.01, extendrect=True)
cbar.set_label(r'Potential Vorticity Units (PVU; $\rm{10^{-6}\ K\ kg^{-1}\ m^{2}\ s^{-1}})$' + ' | 2PVU (dashed)', fontsize=15)


print("\n[+] plotting...")

**************

# **CROSS SECTION ANALYSIS**
<br>

#### 1. Use MetPy's `cross_section` to create variables of "sliced" cross section data
#### 2. Plot cross section analyses of PV and static stability

<br>

## 1. Compute Cross Section Data 

In [ ]:
#################################
# COMPUTE CROSS SECTION USING METPY
#################################
# define the start and end point of the cross section
start = (35, -125.0)
end = (40, -80.0)

# use metpy's `cross_section` to create a new DataSet of cross section "2D" data

cs_data = cross_section(ds, start, end).set_coords(('latitude', 'longitude'))
#cs_sfc_data = cross_section(ds_sfc, start, end).set_coords(('latitude', 'longitude'))


# using the cross section DataSet, calculate the normal component of the wind to the cross section plane
cs_data['nwind'] = mpcalc.normal_component(cs_data['u'], cs_data['v'])

# using metpy compute static stability 
static_stability = mpcalc.first_derivative(cs_data['theta'], axis=0, x=cs_data['isobaricInhPa']*100)

print("\n[+] data preparation complete...")


**************

## 2a. PV Cross Section

In [ ]:
#################################
# CREATE SIMPLE FIGURE
#################################
fig = plt.figure(1, figsize=(14, 10))
ax = plt.subplot(111)


#################################
# PLOT CROSS SECTION DATA
#################################
# plot contourf of PV
pv_contourf = plt.contourf(cs_data['longitude'], cs_data['isobaricInhPa'], cs_data['pv']*1e6, pv_clevs, extend='both', cmap=pv_cmap)

# plot contour of normal-wind to cross section plane, add labels
nwnd_contour = plt.contour(cs_data['longitude'], cs_data['isobaricInhPa'], cs_data['nwind']* 1.94384 , colors='black',linewidths=2)
labels = plt.clabel(nwnd_contour, nwnd_contour.levels[::2], fontsize=10, inline=True,fmt = '%1.0f')
for l in labels:
    l.set_rotation(0)

# plot 2PVU contour


plt.contour(cs_data['longitude'], cs_data['isobaricInhPa'], cs_data['pv']*1e6, [2],
            colors='magenta',linewidths=3,linestyles='dashed')

# plot theta contours
plt.contour(cs_data['longitude'], cs_data['isobaricInhPa'], cs_data['theta'], np.arange(280,500,10),
            colors='white',linewidths=1)


#################################
# MANIPULATE PLOT AXIS, ADD GRID
#################################
ax.grid(True)
ax.set_yscale('symlog')
ax.set_ylim([1000,100])
ax.set_yticklabels(np.arange(1000, 50, -100))
ax.set_yticks(np.arange(1000, 50, -100))

#################################
# ADD PLOT AXIS LABELS AND TITLES
#################################
plt.ylabel('Pressure (hPa)')
plt.xlabel(r'Longitude $\rm{(^\circ E)}$')
plt.title('  RAP Vertical Cross Section\n  Potential Vorticity (PVU), Normal Wind-Component (kts), Potential Temperature (K)', loc='left', fontsize=14)
plt.title(f'{valid_date[0:10]} {valid_date[11:-13]}UTC  ', loc='right', fontsize=14)

#################################
# ADD FILLED CONTOUR COLORBAR
#################################
cbar = plt.colorbar(pv_contourf, aspect=70, fraction=0.02, ax=ax, orientation='horizontal', pad=0.07, extendrect=True)
cbar.set_label(r'Potential Vorticity Units (PVU; $\rm{10^{-6}\ K\ kg^{-1}\ m^{2}\ s^{-1}})$', fontsize=10)

#################################
# CREATE A 2ND AXIS TO PLOT TERRAIN
#################################
# Create the second axis, hide it, set y limit
# plot sfc hghts - 1000hPa hght for a 1000hPa-100hPa yaxis 
ax2 = ax.twinx()
ax2.get_yaxis().set_visible(False)   
ax2.spines['right'].set_visible(False)
ax2.set_ylim([0,15000])
ax2.plot(cs_data['longitude'], cs_data['orog'], color='saddlebrown')
ax2.fill_between(cs_data['longitude'], 0, cs_data['orog'], color='saddlebrown')




#################################
# BUILD AND PLOT MAP INSET
#################################
inset_ax = build_inset_map()
# define map projection
inset_projection = ccrs.LambertConformal()

# PLOT CROSS SECTION POINTS
endpoints = inset_projection.transform_points(ccrs.Geodetic(),*np.vstack([start, end]).transpose()[::-1])
inset_ax.scatter(endpoints[:, 0], endpoints[:, 1], c='k', zorder=2)
inset_ax.plot(endpoints[:,0], endpoints[:,1], c='k', zorder=2, lw=2)

# PLOT 300hPa GEOPOTENTIAL HEIGHTS CONTOUR
cs = inset_ax.contour(ds['longitude'], ds['latitude'], ds['gh'][plev300, :, :], np.arange(0, 18000, 60),
                      colors='black',transform=ccrs.PlateCarree(), linewidths=1)

# PLOT 300hPa PV CONTOURF
inset_contourf = inset_ax.contourf(ds['longitude'], ds['latitude'], ds['pv'][plev300, :, :]*1e6, pv_clevs, cmap=pv_cmap,
                                   transform=ccrs.PlateCarree(), extend='both')

print("\n[+] plotting...")


**************

## 2b. Static stability cross section

In [ ]:
#################################
# CREATE SIMPLE FIGURE
#################################
fig = plt.figure(1, figsize=(14, 10))
ax = plt.subplot(111)


#################################
# PLOT CROSS SECTION DATA
#################################
# plot static stability as a filled contour
static_stability_contourf = plt.contourf(cs_data['longitude'], cs_data['isobaricInhPa'], static_stability*-1,
                                         np.arange(0, 0.015, 0.0005), extend='both', cmap='brg')
for l in labels:
    l.set_rotation(0)


# plot 2PVU contour
plt.contour(cs_data['longitude'], cs_data['isobaricInhPa'], cs_data['pv']*1e6, [2],
            colors='magenta',linewidths=3,linestyles='dashed')

# plot theta contours
plt.contour(cs_data['longitude'], cs_data['isobaricInhPa'], cs_data['theta'], np.arange(280,500,10),
            colors='white',linewidths=1)


#################################
# MANIPULATE PLOT AXIS, ADD GRID
#################################
ax.grid(True)
ax.set_yscale('symlog')
ax.set_ylim([1000,100])
ax.set_yticklabels(np.arange(1000, 50, -100))
ax.set_yticks(np.arange(1000, 50, -100))

#################################
# ADD PLOT AXIS LABELS AND TITLES
#################################
plt.ylabel('Pressure (hPa)')
plt.xlabel(r'Longitude $\rm{(^\circ E)}$')
plt.title('  RAP Vertical Cross Section\n  Static Stability '+r'$\rm{(\frac{K}{hPa})}$'+', Potential Temperature (K), 2PVU', loc='left', fontsize=14)
plt.title(f'{valid_date[0:10]} {valid_date[11:-13]}UTC  ', loc='right', fontsize=14)

#################################
# ADD FILLED CONTOUR COLORBAR
#################################
cbar = plt.colorbar(static_stability_contourf, aspect=70, fraction=0.02, ax=ax, orientation='horizontal', pad=0.07, extendrect=True)
cbar.set_label(r'$\rm{\frac{\partial \theta}{\partial p}\ (\frac{K}{hPa})}$'+' | 2PVU (dashed)', fontsize=12)


#################################
# CREATE A 2ND AXIS TO PLOT TERRAIN
#################################
# Create the second axis, hide it, set y limit
# plot sfc hghts - 1000hPa hght for a 1000hPa-100hPa yaxis 
ax2 = ax.twinx()
ax2.get_yaxis().set_visible(False)   
ax2.spines['right'].set_visible(False)
ax2.set_ylim([0,15000])
ax2.plot(cs_data['longitude'], cs_data['orog'], color='saddlebrown')
ax2.fill_between(cs_data['longitude'], 0, cs_data['orog'], color='saddlebrown')


#################################
# BUILD AND PLOT MAP INSET
#################################
inset_ax = build_inset_map()
# define map projection
inset_projection = ccrs.LambertConformal()

# PLOT CROSS SECTION POINTS
endpoints = inset_projection.transform_points(ccrs.Geodetic(),*np.vstack([start, end]).transpose()[::-1])
inset_ax.scatter(endpoints[:, 0], endpoints[:, 1], c='k', zorder=2)
inset_ax.plot(endpoints[:,0], endpoints[:,1], c='k', zorder=2, lw=2)

# PLOT 300hPa GEOPOTENTIAL HEIGHTS CONTOUR
cs = inset_ax.contour(ds['longitude'], ds['latitude'], ds['gh'][plev300, :, :], np.arange(0, 18000, 60),
                      colors='black',transform=ccrs.PlateCarree(), linewidths=1)

# PLOT 300hPa PV CONTOURF
inset_contourf = inset_ax.contourf(ds['longitude'], ds['latitude'], ds['pv'][plev300, :, :]*1e6, pv_clevs, cmap=pv_cmap,
                                   transform=ccrs.PlateCarree(), extend='both')

<br>

## 3. Plot a plan-view map of the 2PVU isosurface

In [ ]:
#################################
# BUILD MAP FIGURE
#################################
fig, ax = build_map()

if model=='gfs':
    every = 1
    lat=ds['latitude'].values[::every]
    lon=ds['longitude'].values[::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every]
    wind_lon=ds['longitude'].values[::wind_every]
    
elif model=='nam':
    every = 1
    lat=ds['latitude'].values[::every, ::every]
    lon=ds['longitude'].values[::every, ::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every, ::wind_every]
    wind_lon=ds['longitude'].values[::wind_every, ::wind_every]

contourf = ax.contourf(lon, lat, thta_2pvu[::every,::every], np.arange(280,421,3), cmap=theta_2pvu_cmap,
                 transform=ccrs.PlateCarree(), extend='both',alpha=.5)

# plot wind barbs
barbs = ax.barbs(wind_lon,wind_lat,
                 u_2pvu[::wind_every,::wind_every], v_2pvu[::wind_every,::wind_every],
                 length=8, alpha=1.0, transform=ccrs.PlateCarree(), zorder=12)


# add some plot titles
plt.title('  2-PVU-Isosurface Potential Temperature (K), Wind Barbs (kt)', loc='left', fontsize=20)
plt.title(f'{valid_date[0:10]} {valid_date[11:-13]}UTC  ', loc='right', fontsize=20)

# colorbar for filled contour
cbar = plt.colorbar(contourf, aspect=70, fraction=0.02, ax=ax, orientation='horizontal', pad=-0.01, extendrect=True)
cbar.set_label(r'2-PVU-Isosurface Potential Temperature ($\rm{K}$)', fontsize=15)

In [ ]:
#################################
# BUILD MAP FIGURE
#################################
fig, ax = build_map()

if model=='gfs':
    every = 1
    lat=ds['latitude'].values[::every]
    lon=ds['longitude'].values[::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every]
    wind_lon=ds['longitude'].values[::wind_every]
    
elif model=='nam':
    every = 1
    lat=ds['latitude'].values[::every, ::every]
    lon=ds['longitude'].values[::every, ::every]
    wind_every = 20
    wind_lat=ds['latitude'].values[::wind_every, ::wind_every]
    wind_lon=ds['longitude'].values[::wind_every, ::wind_every]

contourf = ax.contourf(lon, lat, p_2pvu[::every,::every], np.arange(50,751,10), cmap=theta_2pvu_cmap,
                 transform=ccrs.PlateCarree(), extend='both',alpha=.5)

# plot wind barbs
barbs = ax.barbs(wind_lon,wind_lat,
                 u_2pvu[::wind_every,::wind_every], v_2pvu[::wind_every,::wind_every],
                 length=8, alpha=1.0, transform=ccrs.PlateCarree(), zorder=12)


# add some plot titles
plt.title('  2-PVU-Isosurface Pressure (hPa), Wind Barbs (kt)', loc='left', fontsize=20)
plt.title(f'{valid_date[0:10]} {valid_date[11:-13]}UTC  ', loc='right', fontsize=20)

# colorbar for filled contour
cbar = plt.colorbar(contourf, aspect=70, fraction=0.02, ax=ax, orientation='horizontal', pad=-0.01, extendrect=True)
cbar.set_label(r'2-PVU-Isosurface Pressure ($\rm{K}$)', fontsize=15)